# AI风险情绪异质性：第一次BERT微调与分析
按顺序运行。运行时请选择 **代码执行程序 → 更改运行时类型 → T4 GPU**。Cookie不要上传到Colab。

In [ ]:
!pip -q install pandas numpy scipy statsmodels scikit-learn torch transformers accelerate

## 1. 上传项目压缩包
把整个项目文件夹压缩为 `ai_risk_project.zip` 后上传。解压后程序会自动寻找包含 `src` 的目录。

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.endswith('.zip'))
!unzip -q -o "$zip_name" -d /content/ai_risk
from pathlib import Path
PROJECT = next(p.parent for p in Path('/content/ai_risk').rglob('src/ai_risk_emotion'))
%cd $PROJECT
print('项目目录：', PROJECT)

## 2. 上传已经仲裁完成的标注CSV
至少包含 `text, anger, fear, root_post_id, event_id`。同一根帖不会跨数据集。

In [ ]:
uploaded = files.upload()
label_file = next(iter(uploaded))
!mkdir -p data/processed
!cp "$label_file" data/processed/labeled.csv
!python -m src.ai_risk_emotion.validate_data --input data/processed/labeled.csv
!python -m src.ai_risk_emotion.split_data --input data/processed/labeled.csv --output-dir data/processed/splits --group-column root_post_id

## 3. 微调BERT（500条仅用于跑通流程，正式模型需5,000—8,000条）

In [ ]:
!python -m src.ai_risk_emotion.train --train data/processed/splits/train.csv --validation data/processed/splits/validation.csv --output outputs/bert-base-chinese --epochs 3 --batch-size 16

## 4. 对全部传播节点输出 Anger/Fear 概率

In [ ]:
!python -m src.ai_risk_emotion.batch_predict --input data/interim/openclaw/nodes.csv --model outputs/bert-base-chinese --output data/processed/predictions.csv

## 5. 计算指标与60%/55%/65%稳健性结果

In [ ]:
for threshold in (0.60, 0.55, 0.65):
    out = f'outputs/analysis_{int(threshold*100)}'
    !python -m src.ai_risk_emotion.analyze --posts data/interim/openclaw/posts.csv --nodes data/interim/openclaw/nodes.csv --predictions data/processed/predictions.csv --output-dir {out} --dominance-threshold {threshold}
!zip -qr outputs/analysis_results.zip outputs/analysis_*
files.download('outputs/analysis_results.zip')